### Initial Setup

In [14]:
container = 'docsub-metrics'
client = 'FRS'
env = 'dev'

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 3, Finished, Available, Finished)

In [15]:
from pyspark.sql import SparkSession
from pyspark.sql import Row
import json
from datetime import datetime 
from pyspark.sql.functions import col, lit, struct
from pyspark.sql.types import StructType, StructField, StringType, IntegerType , DateType , BooleanType , DoubleType ,TimestampType,ArrayType,ArrayType,LongType
from pyspark.sql.functions import col
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, BooleanType, DoubleType
spark = SparkSession.builder.appName("json-to-parquet").config("spark.driver.maxResultSize", "-1").getOrCreate()
from pyspark.sql.utils import AnalysisException
from delta.tables import DeltaTable
from delta.tables import *
import pandas as pd
import numpy as np
from pyspark import StorageLevel
from pyspark.sql.functions import col, lit
import time

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 4, Finished, Available, Finished)

from pyspark.sql import SparkSession
 
# Initialize Spark session
spark = SparkSession.builder.getOrCreate()
 
# Lakehouse / database name
lakehouse_db = "PROD_DOCSUB_METRICS"  # Replace with your database name
column_name = "Policy_ref"             # Column to filter on
value_to_delete = "dcc500d5-8f7a-4ed7-93ce-3c58c9f61e44"      # Value to delete
 
# List all tables in the lakehouse
tables = spark.catalog.listTables(lakehouse_db)
 
for table in tables:
    table_name = table.name
    full_table_name = f"{lakehouse_db}.{table_name}"
 
    # Check if the column exists
    df = spark.table(full_table_name)
    if column_name in df.columns:
        print(f"Deleting from {full_table_name} where {column_name} = '{value_to_delete}'")
        spark.sql(f"DELETE FROM {full_table_name} WHERE {column_name} = '{value_to_delete}'")
    else:
        print(f"Skipping {full_table_name} (column '{column_name}' not found)")

query = f'''Truncate Table total_yellow_fields_detailed '''

result = spark.sql(query)

In [16]:
#add this in the start of each stageingzone notebook
#we create a table vacuummetadata for timestamp and clear old tombstone files


from pyspark.sql import SparkSession
from pyspark.sql import Row
from datetime import datetime
from delta.tables import DeltaTable

# Initialize Spark
spark = SparkSession.builder.getOrCreate()

# Metadata table path and name
metadata_table_path = f"Tables/VacuumMetadata"


# Check if metadata table exists
if not DeltaTable.isDeltaTable(spark, metadata_table_path):
    # If not, create it with the initial timestamp (use an older date to ensure the first run)
    initial_timestamp = datetime.strptime("2023-01-01", "%Y-%m-%d")  # Replace with desired date
    spark.createDataFrame([Row(prev_timestamp=initial_timestamp.strftime("%Y-%m-%d"))]) \
         .write.format("delta").save(metadata_table_path)
else:
    print("Metadata table already exists.")



# Read the metadata table to get the previous timestamp
metadata_df = spark.read.format("delta").load(metadata_table_path)
prev_timestamp_str = metadata_df.select("prev_timestamp").collect()[0][0]
prev_timestamp = datetime.strptime(prev_timestamp_str, "%Y-%m-%d")

# Get current timestamp
current_timestamp = datetime.now()
days_difference = (current_timestamp - prev_timestamp).days

# Check if 7 days have passed
if days_difference >= 7:
    print("Running vacuum operation as timestamp difference is 7 days or more.")

    # List tables and run vacuum on each
    tables = spark.catalog.listTables(f'PROD_DOCSUB_METRICS')
    for table in tables:
        table_name = table.name
        print(f"Vacuuming table: {table_name}")
        spark.sql(f"VACUUM {table_name} RETAIN 168 HOURS").show(truncate=False)

    # Update metadata table with the current timestamp
    new_timestamp = current_timestamp.strftime("%Y-%m-%d")
    new_metadata_df = spark.createDataFrame([Row(prev_timestamp=new_timestamp)])

    # Overwrite the metadata table with the new timestamp
    new_metadata_df.write.format("delta").mode("overwrite").save(metadata_table_path)
    print("Metadata timestamp updated.")
else:
    print("Vacuum operation skipped as timestamp difference is less than 7 days.")



StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 5, Finished, Available, Finished)

Metadata table already exists.


Vacuum operation skipped as timestamp difference is less than 7 days.


### Fabric Integration

In [17]:
import json

# File path in your Lakehouse
file_path = f"/lakehouse/default/Files/LandingZone/{container}/{client}/{container}_{client}.json"

# Open the file and read it
with open(file_path, 'r',encoding='utf-8-sig') as f:
    landing_data = f.read()

# print(landing_data)

# # Split the file contents into separate lines if it's a multi-line JSON file
# # landing_data_list = landing_data.strip().split('\n')

landing_data_list=json.loads(landing_data)

# # Print length to verify that the data is loaded
print(len(landing_data_list))

landing_data=landing_data_list
len(landing_data)
storage_location="Tables"

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 6, Finished, Available, Finished)

750


### Merge Logic

In [18]:
def replace_nan_and_empty_list(value):
    if isinstance(value, list) and len(value) == 0:
        return None
    elif isinstance(value, (np.ndarray, pd.Series)):
        if pd.isna(value).all():
            return None
        else:
            return value
    elif isinstance(value, (float, int, str, type(None))):  # scalar types
        if pd.isna(value):
            return None
        else:
            return value
    else:
        return value

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 7, Finished, Available, Finished)

In [19]:
import pandas as pd

def merge_dataframes_fillna(df1, df2, on_columns, how='outer'):
    """
    Merges two DataFrames on the specified columns, fills missing values
    in overlapping columns, and removes redundant columns with suffixes.

    Parameters:
    df1 (pd.DataFrame): The first DataFrame.
    df2 (pd.DataFrame): The second DataFrame.
    on_columns (list): The column names to merge on.
    how (str): Merge method ('outer', 'inner', etc.).

    Returns:
    pd.DataFrame: The merged DataFrame with missing values handled.
    """
    
    # Merge the DataFrames on the given list of columns
    merged_data = pd.merge(df1, df2, on=on_columns, how=how, suffixes=('_x', '_y'))

    # Identify all columns except the merge key columns
    common_columns = set(merged_data.columns) - set(on_columns)

    # Loop through and fill missing values using the '_y' column first, then '_x'
    for col in common_columns:
        if col.endswith('_x'):
            col_name = col[:-2]  # Remove '_x' to get the base column name
            merged_data[col_name] = merged_data[f'{col_name}_y'].fillna(merged_data[f'{col_name}_x'])

    # Drop the extra columns with '_x' and '_y' suffixes
    merged_data = merged_data.drop(columns=[col for col in merged_data.columns if col.endswith(('_x', '_y'))])

    return merged_data



from pyspark.sql.functions import col, lit
from delta.tables import DeltaTable
from functools import reduce
import operator
import traceback

def handle_missing_columns(existing_df, data):
    """
    Helper function to add missing columns with proper null handling.
    """
    for column in existing_df.columns:
        if column not in data.columns:
            # Add missing column with appropriate type handling
            data[column] = None  # If you need it as string, you can adjust it like below
            # data[column] = data[column].astype(str)

        # If the entire column in data is null, cast it to string type for consistency
        # if data[column].isna().sum() == data.shape[0]:
        #     data[column] = data[column].astype(str)
    
    return data

def mergeandinsert(table_name, data, key_columns):
    delta_table_path = f"{storage_location}/{table_name}"

    if data.empty==True:
        print("Skip since empty")
        return
    
    columnss=[]
    for i in data.columns:
        if data[i].isna().sum()==data.shape[0]:
            columnss.append(i)
    colus=set(data.columns)-set(columnss)
    data=data.loc[:,list(colus)]        
        #     data[i]=data[i].astype(str)

    data=data.replace({np.nan:None,'NaN':None,'nan':None,np.NaN:None,float('inf'): None})    

    data = data.map(replace_nan_and_empty_list)

    try:
        delta_table = DeltaTable.forPath(spark, delta_table_path)
    except AnalysisException:
        delta_table = None
    
    if delta_table is None:
        dfs = spark.createDataFrame(data)
        dfs = dfs.na.replace([float('nan'),float('inf')], [None,None])

        dfs.write.format('delta').save(delta_table_path)
    else:
        try:
            existing_df = spark.read.format("delta").load(delta_table_path).toPandas()

            data = handle_missing_columns(existing_df, data)
            schema = delta_table.toDF().schema
            dfs = spark.createDataFrame(data,schema)
            dfs = dfs.na.replace([float('nan'),float('inf')], [None,None])

            merge_condition = reduce(operator.and_, [col(f"df.{col_name}") == col(f"old_data.{col_name}") for col_name in key_columns])

            delta_table.alias("old_data").merge(dfs.alias("df"), merge_condition) \
                .whenMatchedUpdateAll() \
                .whenNotMatchedInsertAll() \
                .execute()

        except Exception as e:
            print(f"Exception occurred: {e}")
            print(traceback.format_exc())  # Print the full stack trace for better debugging
            existing_df = spark.read.format("delta").load(delta_table_path).toPandas()

            try:
                data = handle_missing_columns(existing_df, data)
                data1=merge_dataframes_fillna(existing_df,data,key_columns)
                existing_dfs = spark.createDataFrame(data1)
                existing_dfs = existing_dfs.na.replace([float('nan'),float('inf')], [None,None])
                existing_dfs.write.format("delta").mode("overwrite").option("overwriteSchema", "True").save(delta_table_path)
            
            except:
                data = handle_missing_columns(existing_df, data)
                existing_df = handle_missing_columns(data, existing_df)

                existing_dfs = spark.createDataFrame(existing_df)
                existing_dfs = existing_dfs.na.replace([float('nan'),float('inf')], [None,None])
                existing_dfs.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(delta_table_path)
                
                delta_table = DeltaTable.forPath(spark, delta_table_path)
                dfs = spark.createDataFrame(data)

                merge_condition = reduce(operator.and_, [col(f"df.{col_name}") == col(f"old_data.{col_name}") for col_name in key_columns])
                
                delta_table.alias("old_data").merge(dfs.alias("df"), merge_condition) \
                    .whenMatchedUpdateAll() \
                    .whenNotMatchedInsertAll() \
                    .execute()


StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 8, Finished, Available, Finished)

### Primary key 

In [20]:
def policy_ref(data):
    id = data['id']
    return str(id)

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 9, Finished, Available, Finished)

### DOCSUB-METRICS 

#### total_green_fields_detailed

In [ ]:
dr= []
for data in landing_data:
    ref = policy_ref(data)
    if 'total_green_fields_detailed' in data and data['total_green_fields_detailed']:
        id = 1
        for jd in data['total_green_fields_detailed']:
            jd={'Policy_ref':ref,'total_green_fields_detailed_Id':id,**jd}
            id+=1
            dr.append(jd)
df = pd.json_normalize(dr)


StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 10, Finished, Available, Finished)

In [ ]:
mergeandinsert('total_green_fields_detailed',df,['total_green_fields_detailed_Id','Policy_ref'])

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 11, Finished, Available, Finished)

#### total_yellow_fields_detailed

In [ ]:
dr= []
for data in landing_data:
    ref = policy_ref(data)
    if 'total_yellow_fields_detailed' in data and data['total_yellow_fields_detailed']:
        id = 1
        for jd in data['total_yellow_fields_detailed']:
            jd={'Policy_ref':ref,'total_yellow_fields_detailed_Id':id,**jd}
            id+=1
            dr.append(jd)
df = pd.json_normalize(dr)

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 12, Finished, Available, Finished)

In [ ]:
mergeandinsert('total_yellow_fields_detailed',df,['total_yellow_fields_detailed_Id','Policy_ref'])

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 13, Finished, Available, Finished)

#### total_modified_green_fields_detailed

In [ ]:
dr= []
for data in landing_data:
    ref = policy_ref(data)
    if 'total_modified_green_fields_detailed' in data and data['total_modified_green_fields_detailed']:
        id = 1
        for jd in data['total_modified_green_fields_detailed']:
            jd={'Policy_ref':ref,'total_modified_green_fields_detailed_Id':id,**jd}
            id+=1
            dr.append(jd)
df = pd.json_normalize(dr)

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 14, Finished, Available, Finished)

In [ ]:
mergeandinsert('total_modified_green_fields_detailed',df,['total_modified_green_fields_detailed_Id','Policy_ref'])

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 15, Finished, Available, Finished)

#### total_modified_yellow_fields_detailed

In [ ]:
dr= []
for data in landing_data:
    ref = policy_ref(data)
    if 'total_modified_yellow_fields_detailed' in data and data['total_modified_yellow_fields_detailed']:
        id = 1
        for jd in data['total_modified_yellow_fields_detailed']:
            jd={'Policy_ref': ref,'total_modified_yellow_fields_detailed_Id':id,**jd}
            id+=1
            dr.append(jd)
df = pd.json_normalize(dr)

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 16, Finished, Available, Finished)

In [ ]:
mergeandinsert('total_modified_yellow_fields_detailed',df,['total_modified_yellow_fields_detailed_Id','Policy_ref'])

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 17, Finished, Available, Finished)

#### manual_added_fields

In [ ]:
dr= []
for data in landing_data:
    ref = policy_ref(data)
    if 'manual_added_fields' in data and data['manual_added_fields']:
        # id = 1
        for jd in data['manual_added_fields']:
            jsonn = {'Policy_ref': ref,'manual_added_fields_Id':id, 'manual_added_fields':jd}
            id+=1
            dr.append(jsonn)
df = pd.json_normalize(dr)

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 18, Finished, Available, Finished)

In [ ]:
mergeandinsert('manual_added_fields',df,['Policy_ref','manual_added_fields_Id'])

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 19, Finished, Available, Finished)

#### Main Table

In [21]:
data_rows = []

for jsonn in landing_data: 
    ref = policy_ref(jsonn)
    # jsonn = {'Policy_ref': ref, **jsonn}
    jsonn.pop('total_green_fields_detailed',None)
    jsonn.pop('total_yellow_fields_detailed',None)
    jsonn.pop('total_modified_green_fields_detailed',None)
    jsonn.pop('total_modified_yellow_fields_detailed',None)
    jsonn.pop('manual_added_fields',None)
    jsonn = {'Policy_ref': ref, **jsonn}
    data_rows.append(jsonn)
    
# print(data_rows)
df = pd.json_normalize(data_rows)

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 20, Finished, Available, Finished)

In [22]:
display(df)

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 21, Finished, Available, Finished)

SynapseWidget(Synapse.DataFrame, ea18b4eb-5b87-4cf2-a86c-6da0fc01d5cc)

In [23]:
mergeandinsert('docsub-metrics',df,['Policy_ref'])

StatementMeta(, c94a6475-1cd8-4166-9eef-977db66fb396, 22, Finished, Available, Finished)